# Semana 14 – Storytelling del EDA: Diagnóstico y Segmentación del Mercado de Vehículos Usados (AutoTec)

## El Contexto Comercial y la Recuperación de Datos

In [9]:
import os
from dotenv import load_dotenv
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pymongo import MongoClient

load_dotenv()
MONGO_URI = os.getenv("MONGO_URI")

# Spark sin conector MongoDB (evita el error de compatibilidad)
spark = SparkSession.builder.appName("Semana14_Storytelling_AutoTec").getOrCreate()

# Carga directa desde MongoDB con PyMongo → Pandas → Spark
client = MongoClient(MONGO_URI)
db = client["proyecto_bigdata"]

print("Cargando datos limpios...")
df_completos = spark.createDataFrame(
    pd.DataFrame(list(db["Contenedor_Autos_Limpio"].find({}, {"_id": 0})))
)

print("Cargando datos etiquetados (K-Means)...")
df_historico = spark.createDataFrame(
    pd.DataFrame(list(db["datos_etiquetados_kmeans"].find({}, {"_id": 0})))
)

total_registros = df_historico.count()
print(f"ÉXITO: {total_registros} vehículos recuperados.")
df_completos.printSchema()

# Utilidades de formato
def formato_pesos_eje():
    plt.gca().yaxis.set_major_formatter(
        ticker.FuncFormatter(lambda x, pos: f"${x:,.0f}".replace(",", "."))
    )

Cargando datos limpios...
Cargando datos etiquetados (K-Means)...


IndexError: list index out of range

# Acto 1: El Storytelling del EDA (El diagnóstico inicial)

En AutoTec, los equipos comerciales enfrentan un problema concreto: tasaciones imprecisas provocadas por criterios subjetivos y datos desactualizados. Este diagnóstico busca responder con datos quién tiene el mayor peso en el portafolio, cómo se distribuyen los precios por marca, y dónde existen señales de riesgo en la valorización.

## Nivel 1 — Vista Estratégica: Concentración del Portafolio por Marca

**Audiencia:** Director General / Gerencia  
**Pregunta que responde:** ¿Qué marcas dominan nuestra base de vehículos y cuánto representa cada una del total?

In [2]:
# KPI Estratégico: participación de cada marca en el portafolio total
total_sku = df_completos.count()

kpi_estrategico = df_completos.groupBy("marca") \
    .agg(
        F.count("marca").alias("Cantidad_Vehiculos"),
        F.round((F.count("marca") / total_sku) * 100, 2).alias("Participacion_Porcentaje")
    ).orderBy(F.desc("Participacion_Porcentaje"))

print(" [KPI ESTRATÉGICO] - ESTRUCTURA DE PORTAFOLIO POR MARCA")
kpi_estrategico.show()

Py4JJavaError: An error occurred while calling o38.count.
: java.lang.NoClassDefFoundError: Could not initialize class com.mongodb.spark.sql.connector.read.MongoScanBuilder
	at com.mongodb.spark.sql.connector.MongoTable.newScanBuilder(MongoTable.java:125)
	at org.apache.spark.sql.execution.datasources.v2.V2ScanRelationPushDown$$anonfun$createScanBuilder$1.applyOrElse(V2ScanRelationPushDown.scala:58)
	at org.apache.spark.sql.execution.datasources.v2.V2ScanRelationPushDown$$anonfun$createScanBuilder$1.applyOrElse(V2ScanRelationPushDown.scala:56)
	at org.apache.spark.sql.catalyst.trees.TreeNode.$anonfun$transformDownWithPruning$1(TreeNode.scala:461)
	at org.apache.spark.sql.catalyst.trees.CurrentOrigin$.withOrigin(origin.scala:76)
	at org.apache.spark.sql.catalyst.trees.TreeNode.transformDownWithPruning(TreeNode.scala:461)
	at org.apache.spark.sql.catalyst.plans.logical.LogicalPlan.org$apache$spark$sql$catalyst$plans$logical$AnalysisHelper$$super$transformDownWithPruning(LogicalPlan.scala

In [ ]:
df_est_pandas = kpi_estrategico.toPandas()

plt.figure(figsize=(12, 6))
sns.set_theme(style="whitegrid")

ax = sns.barplot(
    x="Participacion_Porcentaje",
    y="marca",
    data=df_est_pandas,
    hue="marca",
    palette="Blues_r",
    legend=False
)

for p in ax.patches:
    ax.annotate(f'{p.get_width():.1f}%',
                (p.get_width() + 0.5, p.get_y() + p.get_height() / 2),
                va='center', ha='left', fontsize=11, fontweight='bold')

plt.title("VISTA ESTRATÉGICA: Concentración del Portafolio por Marca (Frecuencia: Mensual)",
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Participación en el Catálogo (% del Total de Vehículos)", fontsize=12)
plt.ylabel("Marca", fontsize=12)
plt.xlim(0, df_est_pandas["Participacion_Porcentaje"].max() + 10)
plt.tight_layout()
plt.show()

## Nivel 2 — Vista Táctica: Bandas de Precio por Marca

**Audiencia:** Gerente Comercial / Jefe de Tasaciones  
**Pregunta que responde:** ¿Cuál es el rango de precios de cada marca y dónde está nuestra exposición a subvalorar o sobrevalorar?

In [ ]:
# KPI Táctico: dispersión y bandas de precio por marca
kpi_tactico = df_completos.groupBy("marca") \
    .agg(
        F.round(F.min("precio"), 0).alias("Precio_Minimo"),
        F.round(F.avg("precio"), 0).alias("Precio_Promedio"),
        F.round(F.max("precio"), 0).alias("Precio_Maximo"),
        F.round(F.stddev("precio"), 0).alias("Desviacion_Estandar")
    ).orderBy(F.desc("Precio_Promedio"))

print(" [KPI TÁCTICO] - MATRIZ DE VOLATILIDAD Y COMPETITIVIDAD DE PRECIOS POR MARCA")
kpi_tactico.show()

In [ ]:
df_tac_pandas = kpi_tactico.toPandas()

plt.figure(figsize=(14, 7))

plt.vlines(x=df_tac_pandas['marca'],
           ymin=df_tac_pandas['Precio_Minimo'],
           ymax=df_tac_pandas['Precio_Maximo'],
           colors='gray', alpha=0.5, linewidth=2,
           label="Rango de Precios (Min a Max)")

plt.scatter(df_tac_pandas['marca'], df_tac_pandas['Precio_Promedio'],
            color='navy', s=100, label="Precio Promedio", zorder=3)
plt.scatter(df_tac_pandas['marca'], df_tac_pandas['Precio_Minimo'],
            color='green', marker='^', s=60, label="Precio Mínimo", zorder=3)
plt.scatter(df_tac_pandas['marca'], df_tac_pandas['Precio_Maximo'],
            color='red', marker='v', s=60, label="Precio Máximo", zorder=3)

plt.title("VISTA TÁCTICA: Bandas de Competitividad y Volatilidad de Precios por Marca (Frecuencia: Semanal)",
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Marca", fontsize=12)
plt.ylabel("Precio del Vehículo (CLP)", fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.legend(loc="upper right", frameon=True)
plt.grid(True, linestyle="--", alpha=0.6)
formato_pesos_eje()
plt.tight_layout()
plt.show()

## Nivel 3 — Vista Operacional: Alertas de Riesgo por Kilometraje y Antigüedad

**Audiencia:** Supervisor de Tasaciones / Analista  
**Pregunta que responde:** ¿Qué vehículos presentan combinaciones críticas de alto kilometraje y alta antigüedad que elevan el riesgo de tasación incorrecta?

In [ ]:
from pyspark.sql.functions import current_date, year as spark_year

# Construimos antigüedad si no existe
df_op = df_completos.withColumn(
    "antiguedad_auto",
    spark_year(current_date()) - F.col("year")
)

# Umbrales operacionales de riesgo
UMBRAL_KM    = 120000
UMBRAL_ANIOS = 8

# Alertas críticas: vehículos en zona de alta depreciación
kpi_operacional_alertas = df_op.filter(
    (F.col("kilometraje") > UMBRAL_KM) & (F.col("antiguedad_auto") > UMBRAL_ANIOS)
).select("marca", "modelo", "year", "antiguedad_auto", "kilometraje", "precio")

print(f" [KPI OPERACIONAL] - ALERTAS CRÍTICAS DE TASACIÓN (Total alertas: {kpi_operacional_alertas.count()})")
kpi_operacional_alertas.show(10)

In [ ]:
df_op_pandas = df_op.select("marca", "kilometraje", "antiguedad_auto", "precio").toPandas()

fig, ax = plt.subplots(figsize=(12, 7))

# Todos los vehículos
ax.scatter(
    df_op_pandas["antiguedad_auto"],
    df_op_pandas["kilometraje"],
    alpha=0.4, s=60, color="darkgray", label="Vehículos"
)

# Zona de riesgo
zona_peligro = df_op_pandas[
    (df_op_pandas["kilometraje"] > UMBRAL_KM) &
    (df_op_pandas["antiguedad_auto"] > UMBRAL_ANIOS)
]
ax.scatter(
    zona_peligro["antiguedad_auto"],
    zona_peligro["kilometraje"],
    color="crimson", s=100,
    label=" ALERTAS CRÍTICAS (Alta Depreciación)",
    edgecolor="black", linewidth=1.2, zorder=4
)

# Líneas de umbral
ax.axvline(x=UMBRAL_ANIOS, color='red', linestyle='--', alpha=0.7, linewidth=1.5)
ax.axhline(y=UMBRAL_KM,    color='red', linestyle='--', alpha=0.7, linewidth=1.5)

# Zonas coloreadas
xlim = ax.get_xlim()
ylim = ax.get_ylim()
ax.axvspan(UMBRAL_ANIOS, xlim[1], ymin=(UMBRAL_KM - ylim[0]) / (ylim[1] - ylim[0]),
           ymax=1.0, color='#FFEBEE', alpha=0.4, zorder=0)
ax.axvspan(xlim[0], UMBRAL_ANIOS, ymin=0, ymax=(UMBRAL_KM - ylim[0]) / (ylim[1] - ylim[0]),
           color='#E8F5E9', alpha=0.3, zorder=0)

ax.text(UMBRAL_ANIOS + 0.3, ylim[1] * 0.88,
        "ZONA DE RIESGO\n(Alta depreciación)",
        color="#B71C1C", weight="bold", fontsize=11)
ax.text(xlim[0] + 0.2, ylim[0] + ylim[1] * 0.05,
        "ZONA SALUDABLE\n(Baja/Media depreciación)",
        color="#1B5E20", weight="bold", fontsize=11)

ax.set_title("VISTA OPERACIONAL: Matriz de Alertas de Depreciación por Antigüedad y Kilometraje (Frecuencia: Diario)",
             fontsize=13, fontweight='bold', pad=15)
ax.set_xlabel("Antigüedad del Vehículo (años)", fontsize=12)
ax.set_ylabel("Kilometraje Acumulado (km)", fontsize=12)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}".replace(",", ".")))
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

# Acto 2: El Descubrimiento del Mercado (Segmentación K-Means)

La segmentación nos permite ir más allá del diagnóstico: en lugar de analizar vehículo por vehículo, identificamos grupos naturales del mercado según precio y kilometraje. Cada clúster representa un perfil comercial distinto con implicancias directas en la estrategia de tasación.

In [ ]:
# Resumen de clústeres: tamaño, precio promedio y kilometraje promedio por segmento
resumen_clusters = df_historico.groupBy("prediction") \
    .agg(
        F.count("prediction").alias("Cantidad_Vehiculos"),
        F.round((F.count("prediction") / total_registros) * 100, 2).alias("Porcentaje_Mercado"),
        F.round(F.avg("precio"), 0).alias("Precio_Promedio"),
        F.round(F.avg("kilometraje"), 0).alias("Kilometraje_Promedio")
    ).orderBy("prediction")

print("Tabla 1: Resumen de segmentos identificados por K-Means")
resumen_clusters.show()
# Cada clúster representa un perfil de depreciación distinto con implicancias directas en tasación.

In [ ]:
# Visualización de los segmentos en el espacio precio vs kilometraje
df_clusters_pd = df_historico.select("precio", "kilometraje", "prediction").toPandas()

etiquetas = {
    0: "Segmento Económico",
    1: "Segmento Medio",
    2: "Segmento Premium"
    # Ajusta según la cantidad de clústeres que hayas definido
}
df_clusters_pd["Segmento"] = df_clusters_pd["prediction"].map(etiquetas).fillna(
    df_clusters_pd["prediction"].astype(str)
)

plt.figure(figsize=(12, 7))
sns.scatterplot(
    data=df_clusters_pd,
    x="kilometraje",
    y="precio",
    hue="Segmento",
    palette="Set2",
    alpha=0.5,
    s=70
)

plt.title("SEGMENTACIÓN DE MERCADO: Clústeres de Vehículos por Precio y Kilometraje",
          fontsize=14, fontweight='bold', pad=15)
plt.xlabel("Kilometraje Acumulado (km)", fontsize=12)
plt.ylabel("Precio del Vehículo (CLP)", fontsize=12)
plt.gca().xaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{x:,.0f}".replace(",", ".")))
formato_pesos_eje()
plt.legend(title="Segmento de Mercado", loc="upper right")
plt.tight_layout()
plt.show()

In [6]:
import subprocess

# Busca todos los parquet disponibles
result = subprocess.run(
    ["find", "/home/jovyan/work", "-name", "*.parquet", "-type", "f"],
    capture_output=True, text=True
)
print("=== ARCHIVOS PARQUET ===")
print(result.stdout if result.stdout else "No se encontraron .parquet")

# También busca carpetas que puedan contener parquet
result2 = subprocess.run(
    ["find", "/home/jovyan/work", "-name", "part-*", "-type", "f"],
    capture_output=True, text=True
)
print("\n=== CARPETAS CON DATOS (part-*) ===")
print(result2.stdout[:3000] if result2.stdout else "No se encontraron")

=== ARCHIVOS PARQUET ===
No se encontraron .parquet

=== CARPETAS CON DATOS (part-*) ===
No se encontraron


In [8]:
import subprocess

# Ver toda la estructura de carpetas
result = subprocess.run(
    ["find", "/home/jovyan/work", "-type", "f"],
    capture_output=True, text=True
)
print(result.stdout[:5000] if result.stdout else "No hay archivos")

/home/jovyan/work/.env
/home/jovyan/work/.git/COMMIT_EDITMSG
/home/jovyan/work/.git/config
/home/jovyan/work/.git/description
/home/jovyan/work/.git/FETCH_HEAD
/home/jovyan/work/.git/HEAD
/home/jovyan/work/.git/hooks/applypatch-msg.sample
/home/jovyan/work/.git/hooks/commit-msg.sample
/home/jovyan/work/.git/hooks/fsmonitor-watchman.sample
/home/jovyan/work/.git/hooks/post-update.sample
/home/jovyan/work/.git/hooks/pre-applypatch.sample
/home/jovyan/work/.git/hooks/pre-commit.sample
/home/jovyan/work/.git/hooks/pre-merge-commit.sample
/home/jovyan/work/.git/hooks/pre-push.sample
/home/jovyan/work/.git/hooks/pre-rebase.sample
/home/jovyan/work/.git/hooks/pre-receive.sample
/home/jovyan/work/.git/hooks/prepare-commit-msg.sample
/home/jovyan/work/.git/hooks/push-to-checkout.sample
/home/jovyan/work/.git/hooks/sendemail-validate.sample
/home/jovyan/work/.git/hooks/update.sample
/home/jovyan/work/.git/index
/home/jovyan/work/.git/info/exclude
/home/jovyan/work/.git/logs/HEAD
/home/jovyan/wor